In [1]:
import os
import re
import json
import zipfile
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from tqdm import tqdm

print("Libraries imported successfully.")
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Libraries imported successfully.
PyTorch version: 2.10.0+cpu
CUDA available: False


In [2]:
#reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

#device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

#main paths
ZIP_PATH = "caption_data.zip"
EXTRACT_DIR = "caption_data_extracted"

print("Current folder:", os.getcwd())

Using device: cpu
Current folder: C:\Users\gelas\Documents\visual-storyteller


In [3]:
if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(
        "caption_data.zip was not found. Please upload/place caption_data.zip in the same folder as this notebook."
    )

print("Found dataset zip file:", ZIP_PATH)

if not os.path.exists(EXTRACT_DIR):
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    print("Dataset extracted successfully.")
else:
    print("Dataset was already extracted.")

Found dataset zip file: caption_data.zip
Dataset was already extracted.


In [4]:
print("Dataset structure preview:\n")

for root, dirs, files in os.walk(EXTRACT_DIR):
    level = root.replace(EXTRACT_DIR, "").count(os.sep)
    indent = "    " * level
    print(f"{indent}{os.path.basename(root)}/")
    
    sub_indent = "    " * (level + 1)
    for file in files[:10]:
        print(f"{sub_indent}{file}")
    
    if level >= 2:
        break

Dataset structure preview:

caption_data_extracted/
    captions.txt
    Images/
        1000268201_693b08cb0e.jpg
        1001773457_577c3a7d70.jpg
        1002674143_1b742ab4b8.jpg
        1003163366_44323f5815.jpg
        1007129816_e794419615.jpg
        1007320043_627395c3d8.jpg
        1009434119_febe49276a.jpg
        1012212859_01547e3f17.jpg
        1015118661_980735411b.jpg
        1015584366_dfcec3c85a.jpg


In [5]:
#find image files
image_extensions = {".jpg", ".jpeg", ".png"}

all_image_paths = []
for root, dirs, files in os.walk(EXTRACT_DIR):
    for file in files:
        if Path(file).suffix.lower() in image_extensions:
            all_image_paths.append(os.path.join(root, file))

print("Number of image files found:", len(all_image_paths))

if len(all_image_paths) == 0:
    raise FileNotFoundError("No image files were found inside the extracted dataset.")

# Find possible caption text/csv files
possible_caption_files = []
for root, dirs, files in os.walk(EXTRACT_DIR):
    for file in files:
        lower_file = file.lower()
        if lower_file.endswith((".txt", ".csv")) and "caption" in lower_file:
            possible_caption_files.append(os.path.join(root, file))

print("Possible caption files:")
for path in possible_caption_files:
    print(path)

if len(possible_caption_files) == 0:
    raise FileNotFoundError("No caption file was found. Please send me the dataset structure output.")

Number of image files found: 8091
Possible caption files:
caption_data_extracted\captions.txt


In [6]:
caption_file = possible_caption_files[0]

print("Using caption file:", caption_file)
print("\nFirst 10 lines:\n")

with open(caption_file, "r", encoding="utf-8", errors="ignore") as f:
    for i in range(10):
        line = f.readline()
        print(line.strip())

Using caption file: caption_data_extracted\captions.txt

First 10 lines:

image,caption
1000268201_693b08cb0e.jpg,A child in a pink dress is climbing up a set of stairs in an entry way .
1000268201_693b08cb0e.jpg,A girl going into a wooden building .
1000268201_693b08cb0e.jpg,A little girl climbing into a wooden playhouse .
1000268201_693b08cb0e.jpg,A little girl climbing the stairs to her playhouse .
1000268201_693b08cb0e.jpg,A little girl in a pink dress going into a wooden cabin .
1001773457_577c3a7d70.jpg,A black dog and a spotted dog are fighting
1001773457_577c3a7d70.jpg,A black dog and a tri-colored dog playing with each other on the road .
1001773457_577c3a7d70.jpg,A black dog and a white dog with brown spots are staring at each other in the street .
1001773457_577c3a7d70.jpg,Two dogs of different breeds looking at each other on the road .


In [ ]:
def load_captions(caption_file):
    """
    Loads captions from common image-captioning dataset formats.
    Supports formats like:
    image,caption
    image.jpg,A dog is running.
    image.jpg#0 A dog is running.
    image.jpg\tA dog is running.
    """
    
    rows = []
    
    with open(caption_file, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()
    
    #semove empty lines
    lines = [line.strip() for line in lines if line.strip()]
    
    #skip header if it looks like one
    if lines[0].lower().replace(" ", "") in ["image,caption", "image_name,caption", "filename,caption"]:
        lines = lines[1:]
    
    for line in lines:
        image_name = None
        caption = None
        
        #format: image.jpg#0 caption
        if "#" in line and ("jpg" in line.lower() or "png" in line.lower()):
            parts = line.split(maxsplit=1)
            if len(parts) == 2:
                image_part = parts[0]
                image_name = image_part.split("#")[0]
                caption = parts[1]
        
        #Format: image.jpg<TAB>caption
        elif "\t" in line:
            parts = line.split("\t")
            if len(parts) >= 2:
                image_name = parts[0].strip()
                caption = " ".join(parts[1:]).strip()
        
        # Format: image.jpg,caption
        elif "," in line:
            parts = line.split(",", 1)
            if len(parts) == 2:
                image_name = parts[0].strip()
                caption = parts[1].strip()
        
        if image_name is not None and caption is not None:
            rows.append({
                "image": image_name,
                "caption": caption
            })
    
    return pd.DataFrame(rows)

captions_df = load_captions(caption_file)

print("Captions loaded:", len(captions_df))
print("Unique images in captions:", captions_df["image"].nunique())
captions_df.head(10)

In [ ]:
# create dictionary: image filename -> full path
image_path_dict = {}

for path in all_image_paths:
    filename = os.path.basename(path)
    image_path_dict[filename] = path

print("Image path dictionary size:", len(image_path_dict))

#add full image paths to dataframe
captions_df["image_path"] = captions_df["image"].map(image_path_dict)

# Remove rows where image path was not found
missing_paths = captions_df["image_path"].isna().sum()
print("Missing image paths:", missing_paths)

captions_df = captions_df.dropna(subset=["image_path"]).reset_index(drop=True)

print("Final captions dataframe size:", len(captions_df))
print("Final unique images:", captions_df["image"].nunique())

captions_df.head()

In [ ]:
sample_images = captions_df["image"].drop_duplicates().sample(3, random_state=SEED).tolist()

for img_name in sample_images:
    img_path = image_path_dict[img_name]
    img = Image.open(img_path).convert("RGB")
    
    related_captions = captions_df[captions_df["image"] == img_name]["caption"].tolist()
    
    plt.figure(figsize=(5, 5))
    plt.imshow(img)
    plt.axis("off")
    plt.title(img_name)
    plt.show()
    
    print("Captions:")
    for cap in related_captions:
        print("-", cap)
    print("\n" + "-" * 80 + "\n")